In [ ]:
!pip install pandas scikit-learn

# Montagem opcional do Google Drive para execução no Google Colab.
import importlib

try:
    google_colab = importlib.import_module('google.colab')
    google_colab.drive.mount('/content/drive')
except (ImportError, AttributeError):
    # Permite executar localmente no VS Code sem o pacote do Colab.
    pass



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
# IMPORTAÇÕES
from pathlib import Path

import pandas as pd
from sklearn import model_selection


In [3]:
# O arquivo pode estar no Google Drive ou no diretório local do projeto.
DATA_PATHS = [
    Path('/content/drive/MyDrive/fatec-ml-semma/results/olist_orders_dataset.csv'),
    Path('/content/drive/MyDrive/data/olist_orders_dataset.csv'),
    Path('./olist_orders_dataset.csv'),
]
DATA_PATH = next((path for path in DATA_PATHS if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        'CSV não encontrado. Envie olist_orders_dataset.csv para a pasta results do projeto no Drive.'
    )

orders = pd.read_csv(DATA_PATH, low_memory=False)
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

print(f'Arquivo: {DATA_PATH}')
print(f'Linhas: {len(orders):,} | Colunas: {orders.shape[1]}')
orders.head()


Arquivo: olist_orders_dataset.csv
Linhas: 99,441 | Colunas: 8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


# Sample

A etapa **S (Sample)** do SEMMA extrai e estrutura uma amostra limpa e representativa antes da modelagem. Neste projeto, a base pública **Brazilian E-Commerce Public Dataset by Olist**, disponível no Kaggle, é usada porque possui datas de observação para cada pedido.

A variável `dtRef` representa o mês da compra. O último mês observado é separado como base **OOT (Out of Time)**; os meses anteriores formam a base histórica usada no desenvolvimento. Assim, a avaliação respeita a ordem temporal e evita usar observações futuras no treino.


In [ ]:
# SAMPLE--------------------------------------------------------
# Base analítica em nível de pedido, com uma referência temporal mensal.
abt = pd.DataFrame({
    'dtRef': orders['order_purchase_timestamp'].dt.to_period('M').astype(str),
    'purchase_year': orders['order_purchase_timestamp'].dt.year,
    'purchase_month': orders['order_purchase_timestamp'].dt.month,
    'purchase_dayofweek': orders['order_purchase_timestamp'].dt.day_of_week,
    'purchase_hour': orders['order_purchase_timestamp'].dt.hour,
    'estimated_delivery_days': (
        orders['order_estimated_delivery_date'] - orders['order_purchase_timestamp']
    ).dt.total_seconds() / (24 * 60 * 60),
    'approval_delay_hours': (
        orders['order_approved_at'] - orders['order_purchase_timestamp']
    ).dt.total_seconds() / (60 * 60),
    'has_carrier_date': orders['order_delivered_carrier_date'].notna().astype('int8'),
    'has_customer_delivery_date': orders['order_delivered_customer_date'].notna().astype('int8'),
    # Target binária para manter a estrutura do exemplo de churn.
    'flagCanceled': orders['order_status'].eq('canceled').astype('int8'),
})

# BASE OOT------------------------------------------------------
reference_month = abt['dtRef'].max()
oot = abt[abt['dtRef'] == reference_month].copy()
df_train = abt[abt['dtRef'] < reference_month].copy()

assert len(oot) > 0
assert len(df_train) > 0
assert reference_month > df_train['dtRef'].max()

# Variáveis explicativas: sem identificadores, status bruto ou a referência temporal.
features = [
    'purchase_year',
    'purchase_month',
    'purchase_dayofweek',
    'purchase_hour',
    'estimated_delivery_days',
    'approval_delay_hours',
    'has_carrier_date',
    'has_customer_delivery_date',
]
target = 'flagCanceled'

X = df_train[features].copy()
X = X.fillna(X.median(numeric_only=True))
y = df_train[target]

# Train/Test split--------------------------------------------------------------
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.2,
    stratify=y,
)

# Exporta as bases produzidas pela etapa S para reuso na modelagem.
output_dir = DATA_PATH.parent
train_path = output_dir / 'olist_orders_sample_train.csv'
oot_path = output_dir / 'olist_orders_sample_oot.csv'
df_train.to_csv(train_path, index=False)
oot.to_csv(oot_path, index=False)

print(f'Mês de referência OOT: {reference_month}')
print(f'Linhas históricas: {len(df_train):,} | Linhas OOT: {len(oot):,}')
print(f'Período histórico: {df_train["dtRef"].min()} a {df_train["dtRef"].max()}')
print(f'Treino: {len(X_train):,} | Teste: {len(X_test):,}')
print(f'Taxa de cancelamento histórica: {y.mean():.3%}')
print(f'Taxa de cancelamento OOT: {oot[target].mean():.3%}')
print(f'Bases exportadas: {train_path} e {oot_path}')
print('Features:', features)


/tmp/ipykernel_12952/2848738283.py:7: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  'purchase_dayofweek': orders['order_purchase_timestamp'].dt.dayofweek,


Mês de referência OOT: 2018-10
Linhas históricas: 99,437 | Linhas OOT: 4
Período histórico: 2016-09 a 2018-09
Treino: 79,549 | Teste: 19,888
Taxa de cancelamento histórica: 0.625%
Taxa de cancelamento OOT: 100.000%
Bases exportadas: olist_orders_sample_train.csv e olist_orders_sample_oot.csv
Features: ['purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'estimated_delivery_days', 'approval_delay_hours', 'has_carrier_date', 'has_customer_delivery_date']
